# LoRA style fine-tune — INPT Smart ICT

Run this on a **GPU runtime** (Colab: Runtime → Change runtime type → T4 GPU).
It LoRA-fine-tunes a small instruct model on `style_dataset.jsonl` (produced by
`finetune/build_dataset.py`) to anchor the answer **style**, then exports the
merged model to **GGUF** for Ollama.

Upload `style_dataset.jsonl` to the Colab session before running.

In [ ]:
%pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install -q --no-deps trl peft accelerate bitsandbytes datasets

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ = 4096
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",  # match your Ollama base
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
)

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files="style_dataset.jsonl", split="train")

def to_text(ex):
    return {"text": tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False)}

ds = ds.map(to_text)
print(ds[0]["text"][:600])

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=2,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=1,
        optim="adamw_8bit",
        output_dir="outputs",
    ),
)
trainer.train()

## Export to GGUF and load into Ollama

Download the produced `*.gguf`, then on your machine:
```
ollama create inpt-smart-ict-ft -f Modelfile.ft
```
where `Modelfile.ft` begins with `FROM ./model-unsloth.Q4_K_M.gguf` plus the
SYSTEM/PARAMETER blocks from `Modelfile.inpt`. Then set
`OLLAMA_MODEL="inpt-smart-ict-ft"` in `.env`.

In [ ]:
model.save_pretrained_gguf("model", tokenizer, quantization_method="q4_k_m")